# Séparation de sources par Deep Learning

In [ ]:
import numpy as np
np.float_ = np.float64  # Musdb n'utilise pas la dernière MàJ de Numpy
import scipy.signal
import librosa
import sklearn.model_selection

## Dataset

Nous allons utiliser le dataset [MUSDB18](https://sigsep.github.io/datasets/musdb.html), qui contient 150 morceaux de musique ainsi qu'une séparation en 4 pistes (basse, batterie, chant et autre). Ces données sont compréssées, échantillonées à $44kHz$ et en stéréo. La librairie Python [`musdb`](https://github.com/sigsep/sigsep-mus-db) permet de manipuler ces données simplement.

Afin d'accélerer les temps de calcul (pour avoir un ordre de grandeur : Deezer a entraîné son réseau pendant plusieurs semaines, et pas avec un compte Google Colab gratuit), nous allons simplifier la tâche de plusieurs manières :
- Au lieu d'entraîner notre réseau sur une centaine de chansons puis de le valider avec les cinquante autres, nous allons entraîner un "mini-réseau" sur le début d'une chanson et valider son apprentissage sur la fin de cette chanson
- Les données seront converties en mono et ré-échantillonées à $22050Hz$
- Nous essaierons ici de séparer uniquement le chant de l'accompagnement (tâche plus simple qu'une séparation en 4).


In [ ]:
# Librairie de manipulation des données
!pip install -q musdb

# Un extrait du dataset
!git clone https://github.com/hugo-paugesteros/musdb.git

In [ ]:
import musdb
mus = musdb.DB(root='/content/musdb', subsets='train')

### Préparation des données

In [ ]:
SR = 22050
MONO = True
FRAME_SIZE = 1024
HOP_SIZE = 1/2 # Ratio of FRAME_SIZE
INPUT_SIZE = (512, 128, 1)

In [ ]:
def preprocess(y, mono, sr):
    if mono :
        y = 0.5 * (y.sum(axis=1, keepdims=True))
    y = librosa.resample(y, orig_sr=44100, target_sr=SR, axis=0)
    return y

def reshape(X):
    X = X[:-1, :-(X.shape[1] % INPUT_SIZE[1])]
    X = X.swapaxes(0, 1)
    X = np.reshape(X, (-1, INPUT_SIZE[1], FRAME_SIZE//2) + X.shape[2:])
    X = X.swapaxes(1, 2)
    return X

x = []
y = []
for track in mus:
    x.append(preprocess(track.audio, MONO, SR))
    tmp = np.stack((
        track.targets['vocals'].audio,
        track.targets['bass'].audio + track.targets['drums'].audio + track.targets['other'].audio
    ), axis=2)
    y.append(preprocess(tmp, MONO, SR))

x = np.concatenate(x, 0)    # (length, channels)
y = np.concatenate(y, 0)    # (length, channels, sources)

_, _, x = scipy.signal.stft(x, nperseg=FRAME_SIZE, noverlap=round((1-HOP_SIZE)*FRAME_SIZE), axis=0)     # (frequencies, channels, times)
_, _, y = scipy.signal.stft(y, nperseg=FRAME_SIZE, noverlap=round((1-HOP_SIZE)*FRAME_SIZE), axis=0)     # (frequencies, channels, sources, times)

x = x.transpose([0, 2, 1])      # (frequencies, times, channels)
y = y.transpose([0, 3, 1, 2])   # (frequencies, times, channels, sources)

x = reshape(x)
y = reshape(y)

x, x_angle, y = (np.abs(x), np.angle(x), np.abs(y))
x_norm = x / x.max()

In [ ]:
x_train, x_val, x_norm_train, x_norm_val, x_train_angle, x_val_angle, y_train, y_val = sklearn.model_selection.train_test_split(x, x_norm, x_angle, y, shuffle=False, test_size=0.1)

## Modèle

Nous allons implémenter une version simplifiée du réseau présenté [`dans cet article`](https://archives.ismir.net/ismir2017/paper/000171.pdf), qui est un U-Net dont l'objectif est de prédire des masques de séparation.

Voici l'architecture du réseau :

![](https://external-content.duckduckgo.com/iu/?u=https%3A%2F%2Fai2-s2-public.s3.amazonaws.com%2Ffigures%2F2017-08-08%2F83ea11b45cba0fc7ee5d60f608edae9c1443861d%2F3-Figure1-1.png&f=1&nofb=1&ipt=9c85949cc6e5069b37ec75b572a29818799b836f8ad8ead36b4c785aac4f465f&ipo=images)

Notre mini-réseau sera composée uniquement des trois premières couches et du "bottleneck" (partie tout en bas du réseau, jonction de la partie descendante et de la partie ascendante).

- chaque bloc de descente sera constitué :
  - d'une couche de convolution 2D, avec un *stride* de 2 (ce qui a pour effet de diviser la taille de l'image par 2) et un noyau de taille 5
  - d'une couche de *BatchNormalization*
  - d'une activation de type *LeakyRelu* avec un paramètre $\alpha = 0.2$
- chaque bloc de remontée sera constitué :
  - d'une couche de déconvolution 2D ($conv2DTranspose$), avec un *stride* de 2 (ce qui a pour effet de multiplier la taille de l'image par 2) et un noyau de taille 5
  - d'une couche de *BatchNormalization*
  - d'une couche de *Dropout* avec un facteur de 0.4
  - d'une couche de concaténation avec la sortie du bloc de descente correspondant
  - d'une activation de type *Relu*
- enfin, le bloc de sortie est constitué
  - d'une couche déconvolution 2D ($conv2DTranspose$) avec autant de filtres que d'instruments à séparer (ici 2)
  - d'une couche de multiplication entre ces filtres et le spectrogramme passé en entrée du réseau



In [ ]:
import tensorflow as tf

In [ ]:
N_LAYERS = 3
CONV_FILTERS = [8, 16, 32, 64]
KERNEL_INITIALIZER = 'uniform'
DROPOUT = 0.4    # Vous pourrez changer ce paramètre

EPOCHS = 1000
BATCH_SIZE = 32
LEARNING_RATE = 1e-3

In [ ]:
inputs = tf.keras.layers.Input(shape=INPUT_SIZE)
inputs = tf.keras.layers.Lambda(lambda x: tf.keras.backend.expand_dims(x, axis=-1))(inputs)
inputs_norm = tf.keras.layers.Input(shape=INPUT_SIZE)

### À compléter ###

x = inputs_norm
# x = tf.keras.layers.GaussianNoise(0.1)(x) #ajout de bruit pour augmenter la robustesse (~data augmentation)
skips = []

# Descente
for i in range(N_LAYERS):
    x = tf.keras.layers.Conv2D(CONV_FILTERS[i], kernel_size=5, strides=2, padding='same', kernel_initializer=KERNEL_INITIALIZER)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.LeakyReLU(alpha=0.2)(x)
    skips.append(x)

# Bottleneck
x = tf.keras.layers.Conv2D(CONV_FILTERS[N_LAYERS], kernel_size=5, strides=2, padding='same', kernel_initializer=KERNEL_INITIALIZER)(x)
x = tf.keras.layers.BatchNormalization()(x)
x = tf.keras.layers.LeakyReLU(alpha=0.2)(x)

# Remontee
for i in reversed(range(N_LAYERS)):
    x = tf.keras.layers.Conv2DTranspose(CONV_FILTERS[i], kernel_size=5, strides=2, padding='same', kernel_initializer=KERNEL_INITIALIZER)(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(DROPOUT)(x)

    x = tf.keras.layers.Concatenate()([x, skips[i]])
    x = tf.keras.layers.ReLU()(x)

outputs = tf.keras.layers.Conv2DTranspose(2, kernel_size=5, strides=2, padding='same', activation='relu', kernel_initializer=KERNEL_INITIALIZER)(x)
outputs = tf.keras.layers.Lambda(lambda x: tf.keras.backend.expand_dims(x, axis=-2))(outputs)
outputs = tf.keras.layers.Multiply()([outputs, inputs])

model = tf.keras.Model(inputs=[inputs_norm, inputs], outputs=outputs)

In [ ]:
print(model.summary())

## Entraînement

In [ ]:
model.compile(
    loss='mae',
    optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
)

model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath=f'checkpoints/best.weights.h5',
    save_weights_only=True,
    monitor='loss',
    mode='min',
    save_best_only=True
)

early_stopping_callback = tf.keras.callbacks.EarlyStopping(
    monitor='loss',
    patience=50,
    verbose=1
)

# reduction du lr quand la loss stagne
reduce_lr_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.8,
    patience=40,
    verbose=1,
    min_lr=1e-6
)

history = model.fit(
    [x_norm_train, x_train], y_train,
    validation_data=([x_norm_val, x_val], y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[model_checkpoint_callback, early_stopping_callback, reduce_lr_callback]
)

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.show()

## Prédictions

In [ ]:
def to_wav_spleeter(X, angle):
    frame_size = X.shape[1]*2
    X = X * np.exp(1j * angle)
    X = np.concatenate(X, axis=1)
    X = np.pad(X, ((0,1), (0,0), (0,0)))
    _, X = scipy.signal.istft(X, nperseg=frame_size, noverlap=round((1-HOP_SIZE)*frame_size), freq_axis=0, time_axis=1)
    return X/np.abs(X).max()

In [ ]:
import IPython.display as ipd

model.load_weights('checkpoints/best.weights.h5')

x_test, x_norm_test, x_test_angle, y_test = x_train, x_norm_train, x_train_angle, y_train
pred = model.predict([x_norm_test, x_test])

y_sum = to_wav_spleeter(pred.sum(axis=-1), x_test_angle)
print('Mixture :')
ipd.display(ipd.Audio(y_sum.T, rate=SR))
for i, label in enumerate(['vocals', 'other']):
    y_true = to_wav_spleeter(y_test[..., i], x_test_angle)
    print(f'Groundtruth - {label} :')
    ipd.display(ipd.Audio(y_true.T, rate=SR))

    y_pred = to_wav_spleeter(pred[..., i], x_test_angle)
    print(f'Prediction - {label} :')
    ipd.display(ipd.Audio(y_pred.T, rate=SR))

In [ ]:
import IPython.display as ipd

model.load_weights('checkpoints/best.weights.h5')

x_test, x_norm_test, x_test_angle, y_test = x_train, x_norm_train, x_train_angle, y_train
# x_test, x_norm_test, x_test_angle, y_test = x_val, x_norm_val, x_val_angle, y_val
pred = model.predict([x_norm_test, x_test])

y_sum = to_wav_spleeter(pred.sum(axis=-1), x_test_angle)
print('Mixture :')
ipd.display(ipd.Audio(y_sum.T, rate=SR))
for i, label in enumerate(['vocals', 'other']):
    y_true = to_wav_spleeter(y_test[..., i], x_test_angle)
    print(f'Groundtruth - {label} :')
    ipd.display(ipd.Audio(y_true.T, rate=SR))

    y_pred = to_wav_spleeter(pred[..., i], x_test_angle)
    print(f'Prediction - {label} :')
    ipd.display(ipd.Audio(y_pred.T, rate=SR))

## Utilisation d'un modèle pré-entraîné

Spleeter est un réseau entraîné par Deezer basé sur l'architecture que vous venez de voir (seules la taille d'entrée et le nombre de couches diffèrent). Ce réseau a été entraîné pendant plusieurs semaines sur un dataset de 25000 chansons. Voyons comment il s'en sort :

**Note** : Si colab vous demande de relancer le runtime pendant l'installation, dites "Oui". Vous perdrez vos variables (donc finissez tout ce que vous avez à faire sur la partie précédente), mais cela est nécessaire pour des raisons d'incompatibilités entre les versions de TensorFlow.

In [ ]:
!pip install spleeter
!pip install librosa

In [ ]:
!spleeter separate -o audio_output musdb/september.mp3

In [ ]:
import librosa
import IPython.display as ipd

y_voc, sr = librosa.load('audio_output/september/vocals.wav')
print(f'Vocals :')
ipd.display(ipd.Audio(y_voc, rate=sr))

y_acc, sr = librosa.load('audio_output/september/accompaniment.wav')
print(f'Other :')
ipd.display(ipd.Audio(y_acc, rate=sr))

![image.png](https://pbs.twimg.com/media/F0msH1XWYAQoIpo?format=png&name=360x360)